- [X] Самостоятельно реализовать функцию **gradient_descent(X, y)**, которая по заданной обучающей выборке обучает модель линейной регрессии, оптимизируя функционал **методом градиентного спуска (Batch Gradient Descent, GD)** и возвращая вектор весов **w**. В качестве функционала можно выбрать, например, функцию ошибок **MSE** + $L_2$-регуляризатор. Использовать матрично-векторные операции для вычисления градиента.
- [X] Найти данные, на которых интересно будет решать задачу регрессии. Зависимость целевого признака от нецелевых должна быть не слишком сложной, чтобы обученная линейная модель смогла показать приемлимый результат. В крайнем случае взять данные для предсказания стоимости машин [тут](https://github.com/rustam-azimov/ml-course-itmo/tree/main/data/car_price) (целевой признак для предсказания --- **price**).
- [X] Считать данные, выполнить первичный анализ данных, при необходимости произвести чистку данных (**Data Cleaning**).
- [X] Выполнить разведочный анализ (**EDA**), использовать визуализацию, сделать выводы, которые могут быть полезны при дальнейшем решении задачи регрессии.
- [X] При необходимости выполнить полезные преобразования данных (например, трансформировать категариальные признаки в количественные), убрать ненужные признаки, создать новые (**Feature Engineering**).
- [X] Случайным образом разбить данные на **обучающую** и **тестовую** выборки, используя методы существующих библиотек.
- [X] При обучении моделей использовать **масштабирование данных**.
- [X] Обучить модель на обучающей выборке, используя функцию **gradient_descent(X, y)**. Оценить качество модели на обучающей и тестовой выборках, используя **MSE**, **RMSE** и $R^2$.
- [X] Обучить модель, используя существующую библиотеку. Например, в **sklearn** для $L_2$-регуляризатора можно использовать **Ridge**. Сравнить качество с Вашей реализацией.
- [X] Повторить тоже самое, но используя **кросс-валидацию**.
- [X] Создать таблицу, со строками (mse-train, mse-test, rmse-train, rmse-test, r2-train, r2-test) и столбцами (Fold1, Fold2, ..., Foldk, E, STD), где k --- количество фолдов в кросс-валидации, **E** --- мат. ожидание и **STD** --- стандартное отклонение. Сделать выводы.
- [X] * (+2 балла) Перед обучением моделей подобрать наилучшее количество (и само подмножество) признаков, например используя **Recursive Feature Elimination (RFE)** из **sklearn**.
- [X] * (+1 балл) Во все ваши реализации добавить возможность настройки нужных **гиперпараметров**, а в процессе обучения **всех** моделей осуществить подбор оптимальных значений этих гиперпараметров.
- [ ] * (+1 балл) Также самостоятельно реализовать **метод стохастического градиентного спуска (Stochastic Gradient Descent, SGD)**, обучить модели и добавить их во все сравнения.
- [ ] * (+1 балл) Также самостоятельно реализовать **метод мини-пакетного градиентного спуска (Mini Batch Gradient Descent)**, обучить модели и добавить их во все сравнения.


In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from scipy import stats

%precision %.4f

from datetime import datetime

from sklearn.metrics import make_scorer, mean_squared_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_validate
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator
from sklearn.linear_model import Ridge
from sklearn.feature_selection import RFECV
from scipy.stats import uniform, randint

In [ ]:
class MyLinearRegression(BaseEstimator):
	def __init__(self, epsilon: float = 0.0001, alpha: float = 1.0, learning_rate: float = 0.01, max_iter: int = 100000):
		self.coef_ = None
		self.alpha = alpha
		self.epsilon = epsilon
		self.max_iter = max_iter
		self.learning_rate = learning_rate
	
	def __gradient_mse_l2_step(self, w, x_train, y_train):
		n_samples = x_train.shape[0]

		dummy_feature = np.ones((n_samples, 1))
		x_prepared = np.hstack((x_train, dummy_feature))

		errors = x_prepared.dot(w) - y_train

		grad_mse = (2.0 / n_samples) * x_prepared.T.dot(errors)
		grad_l2 = 2 * self.alpha * w
		grad_l2[-1] = 0

		return grad_mse + grad_l2

	def __gradient_descent(self, x_train, y_train):
		k = 1
		weights = np.random.randn(x_train.shape[1] + 1)

		for _ in range(self.max_iter):
			w_prev = weights.copy()

			grad = self.__gradient_mse_l2_step(w_prev, x_train, y_train)
			weights = w_prev - self.learning_rate * grad

			if np.linalg.norm(weights - w_prev) < self.epsilon:
				break

			k += 1

		self.coef_ = weights

		return weights
	
	def fit(self, x_train, y_train):
		self.__gradient_descent(x_train, y_train)
		return self

	def predict(self, X):
		x_prepared = np.c_[X, np.ones(X.shape[0])]
		return x_prepared.dot(self.coef_)

	def score(self, X, y):
		"""Method for compatibility with RandomizedSearchCV"""
		y_pred = self.predict(X)
		return -mean_squared_error(y, y_pred)


В качестве набора данных используется [Global Weather Repository](https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository/versions/34) --- ежедневная информация о погоде в столицах разных стран. 

Целевой признак --- feel_temperature_diff_celsius (разница между ощущаемой и фактической температурой в градусах Цельсия).

In [93]:
df = pd.read_csv("../data/GlobalWeatherRepository.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6621 entries, 0 to 6620
Data columns (total 41 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   country                       6621 non-null   object 
 1   location_name                 6621 non-null   object 
 2   latitude                      6621 non-null   float64
 3   longitude                     6621 non-null   float64
 4   timezone                      6621 non-null   object 
 5   last_updated_epoch            6621 non-null   int64  
 6   last_updated                  6621 non-null   object 
 7   temperature_celsius           6621 non-null   float64
 8   temperature_fahrenheit        6621 non-null   float64
 9   condition_text                6621 non-null   object 
 10  wind_mph                      6621 non-null   float64
 11  wind_kph                      6621 non-null   float64
 12  wind_degree                   6621 non-null   int64  
 13  win

В датасете большое количество признаков, по которым сложно или почти невозможно будет предсказать ощущаемую температуру. Также есть избыточные признаки, например, температура в фаренгейтах, wind_mph и др. Заранее удалим timezone, last_updated_epoch, temperature_fahrenheit, moon_phase, wind_mph, wind_direction, pressure_in, precip_in, feels_line_fahrenheit, visibility_miles, gust_mph

In [94]:
df = df.drop(['country', 'location_name', 'timezone', 'last_updated_epoch', 'temperature_fahrenheit', 'moon_phase', 'wind_mph', 'wind_direction', 'pressure_in', 'precip_in', 'feels_like_fahrenheit', 'visibility_miles', 'gust_mph'], axis=1)

Посмотрим, есть ли пропущенные значения

In [95]:
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

Series([], dtype: int64)


В столбцах moonset и moonrise есть значения "No moonset" или "No moonrise". Они означают, что в этот день не было лунного захода или восхода соответственно. Это возможно при следующих обстоятельствах: от первого восхода (захода) до второго восхода (захода) проходит больше 24 часов. Заменим "No moonrise" и "No moonset" на 12:00 AM.

In [96]:
df["moonset"] = df["moonset"].replace("No moonset", "12:00 AM")
df["moonrise"] = df["moonrise"].replace("No moonrise", "12:00 AM")

Преобразуем дату и время

In [97]:
df['last_updated'] = df['last_updated'].apply(lambda time_str: datetime.strptime(time_str, "%Y-%m-%d %H:%M")) # string to datetime object
for column in ["sunrise", "sunset", "moonset", "moonrise"]:
  df[column] = df[column].apply(lambda time_str: datetime.strptime(time_str, "%I:%M %p"))
  df[column] = df[column].dt.hour + df[column].dt.minute / 60

Далее необходимо преобразовать категориальные признаки в количественные

In [98]:
condition_text_unique = df['condition_text'].unique()
print(f'condition_text unique values count: {condition_text_unique.size}')
df['condition_encoded'] = pd.factorize(df['condition_text'])[0]
df = df.drop(['condition_text'], axis=1)

condition_text unique values count: 23


In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6621 entries, 0 to 6620
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   latitude                      6621 non-null   float64       
 1   longitude                     6621 non-null   float64       
 2   last_updated                  6621 non-null   datetime64[ns]
 3   temperature_celsius           6621 non-null   float64       
 4   wind_kph                      6621 non-null   float64       
 5   wind_degree                   6621 non-null   int64         
 6   pressure_mb                   6621 non-null   float64       
 7   precip_mm                     6621 non-null   float64       
 8   humidity                      6621 non-null   int64         
 9   cloud                         6621 non-null   int64         
 10  feels_like_celsius            6621 non-null   float64       
 11  visibility_km                 

Добавим признак diff_temp --- разность между фактической температурой и ощущаемой.

In [100]:
df['diff_temp'] = df['temperature_celsius'] - df['feels_like_celsius']

Посмотрим, как динамика различных признаков коррелирует с динамикой разности температур.

In [101]:
correlations = df.corrwith(df['diff_temp'])
correlations.abs().sort_values(ascending=False)


diff_temp                       1.000000
feels_like_celsius              0.829415
temperature_celsius             0.671203
pressure_mb                     0.354375
uv_index                        0.340250
sunset                          0.264146
sunrise                         0.213147
latitude                        0.199353
cloud                           0.183935
longitude                       0.140681
humidity                        0.126817
precip_mm                       0.098481
gust_kph                        0.082823
wind_kph                        0.082064
visibility_km                   0.073361
condition_encoded               0.067022
air_quality_Nitrogen_dioxide    0.042600
moonrise                        0.041639
air_quality_Ozone               0.039364
moonset                         0.037634
air_quality_Sulphur_dioxide     0.032954
air_quality_gb-defra-index      0.031548
wind_degree                     0.030998
air_quality_us-epa-index        0.017254
air_quality_Carb

Удалим признаки с незначительной корреляцией (до 0.05):

air_quality_Nitrogen_dioxide

moonrise

air_quality_Ozone

moonset

air_quality_Sulphur_dioxide

air_quality_gb-defra-index

wind_degree

air_quality_us-epa-index

air_quality_Carbon_Monoxide

last_updated

moon_illumination

air_quality_PM10

air_quality_PM2.5

Также удалим признак feels_like_celsius, так как он используется при вычислении разницы.

In [102]:
df = df.drop(['air_quality_Nitrogen_dioxide', 'moonrise', 'air_quality_Ozone', 'moonset',
			  'air_quality_Sulphur_dioxide', 'air_quality_gb-defra-index', 'wind_degree',
			  'air_quality_us-epa-index', 'air_quality_Carbon_Monoxide', 'last_updated',
			  'moon_illumination', 'air_quality_PM10', 'air_quality_PM2.5', 'feels_like_celsius'], axis=1)


Для выбора метода масштабирования проведем анализ распределений. Используем критерий согласия Колмогорова-Смирнова для проверки соответствия выборки нормальному распределению.

In [103]:

num_columns = len(df.columns)
plots_per_row = 4
num_rows = (num_columns + plots_per_row - 1) // plots_per_row

for i, column in enumerate(df.columns):
	statistic, p_value = stats.kstest(df[column], 'norm', args=(np.mean(df[column]), np.std(df[column])))
	print(f'{column}: {'normal' if p_value > 0.05 else 'not_normal'}, p-value: {p_value}')


latitude: not_normal, p-value: 4.593006733654758e-53
longitude: not_normal, p-value: 3.092793784170601e-60
temperature_celsius: not_normal, p-value: 5.9181218843567384e-46
wind_kph: not_normal, p-value: 1.8382753777899466e-176
pressure_mb: not_normal, p-value: 3.967928758980429e-31
precip_mm: not_normal, p-value: 0.0
humidity: not_normal, p-value: 9.76623295301717e-72
cloud: not_normal, p-value: 7.756081532004964e-181
visibility_km: not_normal, p-value: 0.0
uv_index: not_normal, p-value: 0.0
gust_kph: not_normal, p-value: 5.644742251571342e-37
sunrise: not_normal, p-value: 3.746876334554871e-27
sunset: not_normal, p-value: 3.548593270317294e-47
condition_encoded: not_normal, p-value: 0.0
diff_temp: not_normal, p-value: 7.760154963092696e-145


Таким образом, маловероятно, что хотя бы один из признаков имеет нормальное распределение. Выполним min-max масштабирование. Также разобьем данные на обучающую и тестовую выборки.

In [104]:
y = df["diff_temp"]
X = df.drop(columns=["diff_temp"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.fit_transform(X_test), columns=X.columns)

rfecv = RFECV(Ridge(), cv=5)
rfecv.fit(X_train_scaled, y_train)
selected_features = X.columns[rfecv.support_]
print("Retained features:", set(selected_features))
print("Removed features:", set(X.columns) - set(selected_features))

X = X[selected_features]
X_train_scaled = X_train_scaled[selected_features]
X_test_scaled = X_test_scaled[selected_features]

Retained features: {'gust_kph', 'precip_mm', 'visibility_km', 'longitude', 'uv_index', 'temperature_celsius', 'humidity', 'sunset', 'sunrise', 'wind_kph'}
Removed features: {'condition_encoded', 'pressure_mb', 'cloud', 'latitude'}


Воспользуемся библиотечным Ridge.

In [105]:
model = Ridge()

model.fit(X_train_scaled, y_train)
y_test_pred = model.predict(X_test_scaled)

print("Test MSE = ", mean_squared_error(y_test, y_test_pred))
print("Test RMSE = ", root_mean_squared_error(y_test, y_test_pred))
print("Test R^2 = ", r2_score(y_test, y_test_pred))

Test MSE =  3.8473235479050083
Test RMSE =  1.961459545314409
Test R^2 =  0.5332769856232269


Воспользуемся самостоятельно реализованным алгоритмом линейной регрессии: обучим модель и предскажем при помощи нее.

In [106]:
my_model = MyLinearRegression()
my_model.fit(X_train_scaled, y_train)
y_test_my_pred = my_model.predict(X_test_scaled)

print("Test MSE = ", mean_squared_error(y_test, y_test_my_pred))
print("Test RMSE = ", root_mean_squared_error(y_test, y_test_my_pred))
print("Test R^2 = ", r2_score(y_test, y_test_my_pred))

Test MSE =  7.895939876622079
Test RMSE =  2.809971508151298
Test R^2 =  0.042134924534345086


Получили удовлетворительные результаты библиотечного Ridge (R^2 > 0.5) и неудовлетворительные результаты собственной реализации линейной регрессии. Попробуем подобрать гиперпараметры при помощи RandomizedSearchCV, количество наборов параметров --- 1000. Также используем кросс валидацию, количество разбиений в процессе кросс валидации --- 5.

In [107]:
folds_k = 5

my_param_dist = {
    'alpha': uniform(0.01, 10),
    'max_iter': randint(1000, 50000),
    'epsilon': uniform(1e-7, 1e-3),
}

lib_param_dist = {
    'alpha': uniform(0.01, 10),
}

my_model = MyLinearRegression()

my_random_search = RandomizedSearchCV(
    estimator=my_model,
    param_distributions=my_param_dist,
    n_iter=1000,
    cv=folds_k,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

lib_model = Ridge()

lib_random_search = RandomizedSearchCV(
    estimator=lib_model,
    param_distributions=lib_param_dist,
    n_iter=1000,
    cv=folds_k,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

estimators = {
	"My Ridge": my_random_search,
    "Lib Ridge": lib_random_search
}

scorings = {
	'mse': make_scorer(mean_squared_error),
	'rmse': make_scorer(root_mean_squared_error),
	'r2': make_scorer(r2_score)
}


Построим таблицу со строками (mse-train, mse-test, rmse-train, rmse-test, r2-train, r2-test) и столбцами (Fold1, Fold2, ..., Foldk, E, STD), где k --- количество фолдов в кросс-валидации, **E** --- мат. ожидание и **STD** --- стандартное отклонение.

In [108]:
for (estimator_name, estimator) in estimators.items():
    table = pd.DataFrame()

    print(estimator_name)

    pipeline = Pipeline([
        ('scaling', MinMaxScaler()),
        ('estimator', estimator),  
    ])

    cv_results = cross_validate(
            estimator=pipeline,
            X=X,
            y=y,
            cv=folds_k,
            scoring=scorings,
            return_train_score=True,
            return_estimator=True
    )

    for metric_name, _ in scorings.items():
        table[f"{metric_name}-train"] = cv_results[f"train_{metric_name}"]
        table[f"{metric_name}-test"] = cv_results[f"test_{metric_name}"]
    
    table = table.transpose()

    table.columns = [f"Fold{k + 1}" for k in range(folds_k)]

    table['E'] = table.mean(axis=1)
    table['STD'] = table.std(axis=1)

    display(table.style.set_caption(estimator_name))

    best_params_list = []

    for fold, pipeline in enumerate(cv_results['estimator'], 1):
        random_search = pipeline.named_steps['estimator']        
        best_params = random_search.best_params_.copy()
        best_params['fold'] = fold
        best_params_list.append(best_params)

    params_df = pd.DataFrame(best_params_list)
    print("Лучшие параметры по фолдам:")
    display(params_df.round(4).style)



My Ridge
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits


,Fold1,Fold2,Fold3,Fold4,Fold5,E,STD
mse-train,4.225891,4.521739,4.292368,4.551456,4.170612,4.352413,0.155534
mse-test,4.785269,3.779805,4.724870,3.626093,5.084127,4.400033,0.584051
rmse-train,2.055697,2.126438,2.071803,2.133414,2.042208,2.085912,0.037205
rmse-test,2.187526,1.944172,2.173677,1.904230,2.254801,2.092881,0.141002
r2-train,0.475789,0.458172,0.469936,0.457458,0.463350,0.464941,0.007028
r2-test,0.431572,0.481125,0.428865,0.488763,0.469299,0.459925,0.025050


Лучшие параметры по фолдам:


,alpha,epsilon,max_iter,fold
0,0.013300,0.000000,38947,1
1,0.013300,0.000000,38947,2
2,0.013300,0.000000,38947,3
3,0.013300,0.000000,38947,4
4,0.013300,0.000000,38947,5


Lib Ridge
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits


,Fold1,Fold2,Fold3,Fold4,Fold5,E,STD
mse-train,3.579631,3.738593,3.520366,3.766475,3.490909,3.619195,0.112904
mse-test,3.858029,3.181629,4.079679,3.080427,4.215506,3.683054,0.466051
rmse-train,1.891991,1.933544,1.876264,1.940741,1.868397,1.902188,0.029622
rmse-test,1.964187,1.783712,2.019821,1.755114,2.053170,1.915201,0.122717
r2-train,0.555956,0.552014,0.565270,0.551029,0.550810,0.555016,0.005452
r2-test,0.541716,0.563240,0.506854,0.565695,0.559969,0.547495,0.021999


Лучшие параметры по фолдам:


,alpha,fold
0,1.718900,1
1,0.680800,2
2,0.643700,3
3,0.845000,4
4,0.812300,5


С использованием кросс-валидации и специального подбора значений параметров результаты модели MyLinearRegression сильно улучшились и стали близкими к результатам библиотечной модели: E[R^2]=0.46.

Ранее была проведена оценка моделей без RFE. Стоит отметить, что удаление признаков при помощи Recursive Feature Elimineation не улучшило результаты модели MyLinearRegression.

Полученные результаты R^2-score на обеих реализациях линейной регрессии (E[R^2]=0.46 и E[R^2]=0.55) свидительствуют о том, что сама модель линейной регрессии не объясняет около 50% дисперсии зависимой переменной.